# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [4]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'kaushik-demo'

### Producer Mode -> Writes Data to Broker

In [7]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# bootstrap_servers is the address of your Kafka broker.
# Via SSH tunnel (class): ['localhost:9092']. Same for local Docker Kafka.
producer = KafkaProducer(bootstrap_servers=['localhost:9092'],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# Cities of your choice
cities = ['checkpoint 1','Chicago', 'New York', 'San Francisco']

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)
producer.flush()
print("Done writing.")

Writing to Kafka Broker
Writing: 2026-09-25 17:05:55,checkpoint 1,30ºC
Writing: 2026-09-25 17:05:56,checkpoint 1,28ºC
Writing: 2026-09-25 17:05:57,San Francisco,23ºC
Writing: 2026-09-25 17:05:58,San Francisco,31ºC
Writing: 2026-09-25 17:05:59,Chicago,30ºC
Writing: 2026-09-25 17:06:00,checkpoint 1,24ºC
Writing: 2026-09-25 17:06:01,San Francisco,22ºC
Writing: 2026-09-25 17:06:02,San Francisco,29ºC
Writing: 2026-09-25 17:06:03,Chicago,31ºC
Writing: 2026-09-25 17:06:04,checkpoint 1,30ºC
Done writing.


### Consumer Mode -> Reads Data from Broker

In [8]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# First arg is the topic to subscribe to. auto_offset_reset='earliest' reads
# from the start; use 'latest' to read only new messages arriving after you start.
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest', #Experiment with different values
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000,
    consumer_timeout_ms=15000 # stop after 15s idle so notebook cell finishes
)

print('Reading Kafka Broker')
for message in consumer:
    message = message.value.decode()
    # Default message.value type is bytes!
    print(loads(message))
    os.system(f"echo {message} >> kafka_log.csv")

Reading Kafka Broker
2026-09-25 17:00:48,Chicago,26ºC
2026-09-25 17:00:50,San Francisco,22ºC
2026-09-25 17:00:51,New York,30ºC
2026-09-25 17:00:52,New York,20ºC
2026-09-25 17:00:53,New York,24ºC
2026-09-25 17:00:54,San Francisco,18ºC
2026-09-25 17:00:55,Chicago,32ºC
2026-09-25 17:00:56,San Francisco,18ºC
2026-09-25 17:00:57,Chicago,31ºC
2026-09-25 17:00:58,New York,20ºC
2026-09-25 17:05:55,checkpoint 1,30ºC
2026-09-25 17:05:56,checkpoint 1,28ºC
2026-09-25 17:05:57,San Francisco,23ºC
2026-09-25 17:05:58,San Francisco,31ºC
2026-09-25 17:05:59,Chicago,30ºC
2026-09-25 17:06:00,checkpoint 1,24ºC
2026-09-25 17:06:01,San Francisco,22ºC
2026-09-25 17:06:02,San Francisco,29ºC
2026-09-25 17:06:03,Chicago,31ºC
2026-09-25 17:06:04,checkpoint 1,30ºC


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset